# Clinical Reasoning Agent — retrieval A/B

Measures what retrieval changes, by running the same five scenarios twice: once with
`retrieved=None` and once with TF-IDF retrieval over the 30-unit corpus.

**What this can establish.** Whether adding retrieval changes the validators' verdicts on
five designed cases, and in which direction.

**What it cannot.** Anything about diagnostic accuracy. There is no clinical ground truth
here. The cases are synthetic and the comparison is against the system's own safety layer,
not against what a clinician would have concluded.

**The frozen `clinical_reasoning_pre_rag` baseline no longer serves as a confound check.**
It was captured against the free-text evidence prompt, so `DIFFERS` is expected on every case
and means nothing is wrong. It is a record of what the system did before. What must still
hold in section 9 is that escalation MATCHES across arms -- it does not pass through the
model, so a difference there would be a bug rather than a result.

**Known limitation, recorded before running.** Unit F20 — what a negative FAST does not
exclude — does not retrieve for its own question. TF-IDF ranks F18 above it because F18
repeats "FAST negative" in a short passage while F20 is long and says "false-negative" and
"sensitivity" instead. If a case fails to improve, this is a candidate explanation.

Runtime: **Runtime → Change runtime type → T4 GPU**. About 15 minutes end to end.

## 1. Runtime

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU"

## 2. llama-cpp-python (CUDA build)

In [ ]:
# Prebuilt CUDA wheel. If this fails, use the fallback cell below.
!pip -q install llama-cpp-python \
    --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124

import llama_cpp
print("llama_cpp", llama_cpp.__version__)

In [ ]:
# FALLBACK ONLY -- run this only if the cell above failed. Compiles from source, ~10 min.
# !CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python --force-reinstall --no-cache-dir

## 3. Model

In [ ]:
!pip -q install huggingface_hub
from pathlib import Path
from huggingface_hub import hf_hub_download

# Downloaded inside Colab rather than uploaded: ~2 minutes on Google's network against
# hours to push 4.6 GB from a home connection. Lands in /content, cleared on runtime
# restart, so this re-runs each session.
MODEL = Path(hf_hub_download(
    repo_id='bartowski/HuatuoGPT-o1-8B-GGUF',
    filename='HuatuoGPT-o1-8B-Q4_K_M.gguf',
    local_dir='/content/model'))

print(f"ready: {MODEL}  ({MODEL.stat().st_size/1e9:.1f} GB)")

## 4. Code

Upload **`pocus_agents_colab.zip`**. It must be the rebuilt one — it carries corpus v1.0
(all 30 units sourced) and the frozen baseline used by section 9.

In [ ]:
from google.colab import files
import zipfile, sys, os

up = files.upload()                       # pick pocus_agents_colab.zip
with zipfile.ZipFile(next(iter(up))) as z:
    z.extractall('/content/pocus')

# Scrub the import path before importing anything. A copy of `src` in Drive takes priority
# otherwise, and Python keeps serving whichever version it imported first -- which once
# produced byte-identical results from "new" code and looked like a finding rather than a
# mistake.
sys.path = [p for p in sys.path if 'POCUS-Project' not in p and 'drive' not in p]
sys.path.insert(0, '/content/pocus')
for m in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[m]

os.environ['POCUS_LLM_PATH'] = str(MODEL)

import src.agents.clinical.reasoning as R
print('loaded from  :', R.__file__)
# Reads the COMPILED function, so a newer file sitting unused on disk cannot fool it.
print('NEW code live:', 'focus' in R.reason.__code__.co_varnames)

## 5. Corpus and retrieval sanity

Costs nothing and runs before the GPU work. Two things it catches: a stale zip, and a
scenario that retrieves nothing — whose two arms would then be identical by construction.

In [ ]:
import json
from src.agents.clinical.retrieval import Retriever, retrieval_note, is_grounded
from src.agents.clinical.run_case import SCENARIOS, build

corpus = json.load(open('/content/pocus/src/agents/clinical/corpus/pocus_corpus.json'))
print('corpus version :', corpus['corpus_version'])
print('sourced        :', sum(p['status'] == 'sourced' for p in corpus['passages']),
      '/', len(corpus['passages']))
assert corpus['corpus_version'] == '1.0-all-units-sourced', 'stale zip -- re-upload'

retriever = Retriever()
for s in SCENARIOS:
    hits = retriever.for_state(build(s))
    print(f"\n{s}")
    print("   ", retrieval_note(hits), "| grounded:", is_grounded(hits))
    for h in hits:
        print(f"     [{h['n']}] {h['id']} {h['topic']:<28} {h['score']:.3f}")

## 6. Safety benchmark — the layer under test, before the model is involved

In [ ]:
!cd /content/pocus && python -m src.agents.tests.run_benchmark

## 7. Load the model

In [ ]:
import time
from src.agents.clinical.llm import LlamaCppBackend

t0 = time.time()
backend = LlamaCppBackend(n_gpu_layers=-1, n_ctx=4096, max_tokens=1400, verbose=False)
print(f"model loaded in {time.time()-t0:.1f}s")

## 8. The A/B run

Arm A (`none`) runs entirely before arm B (`tfidf`), so arm A can be checked against the
frozen baseline as a block rather than interleaved with runs that changed the prompt.

`max_revisions=1`, measured rather than chosen. Two rounds were tried: the second round was
reached and used on three of five cases and the answers came back with the same faults, for
32% more runtime. The capability is real and tested
(`test_two_independent_faults_need_two_revision_rounds`); it is not worth its cost with this
model.

Two faults are gone from this run by construction rather than by persuasion. A comma-joined
`missing_information` is normalised in Python and reported under `normalizations`, and every
abnormal value now reaches the reader through `evidence_considered` whether the model cited it
or not.

In [ ]:
import time, json
from src.agents.clinical.run_case import SCENARIOS, build
from src.agents.clinical.reasoning import reason
from src.agents.clinical.retrieval import Retriever, retrieval_note, is_grounded

retriever = Retriever()
ab = {}

for arm, use_retrieval in (('none', False), ('tfidf', True)):
    ab[arm] = {}
    print(f"\n{'='*70}\nARM: {arm}\n{'='*70}")
    for s in SCENARIOS:
        st = build(s)
        hits = retriever.for_state(st) if use_retrieval else None
        t0 = time.time()
        out = reason(st, llm_fn=backend, retrieved=hits, max_revisions=1)
        dt = time.time() - t0
        out['_seconds'] = dt
        out['_retrieval'] = {
            'passages': len(hits or []),
            'ids': [h['id'] for h in (hits or [])],
            'grounded': is_grounded(hits or []),
            'note': retrieval_note(hits or []),
        }
        ab[arm][s] = out
        print(f"  {s:<14} {dt:6.1f}s  "
              f"withheld={str(out.get('differential_withheld', False)):<5} "
              f"errors={len(out['validation_errors'] or [])}  "
              f"warnings={len(out.get('warnings') or [])}  "
              f"revisions={len(out.get('revisions') or [])}  "
              f"hits={out['_retrieval']['ids']}")

json.dump(ab, open('/content/ab_retrieval.json', 'w'), indent=2, default=str)
print('\nsaved /content/ab_retrieval.json')

## 9. Comparison

The first block is the one that can invalidate everything else.

In [ ]:
import json
base = json.load(open('/content/pocus/models/clinical_reasoning_pre_rag/results.json'))

# reason() sets validation_errors to None -- not [] -- when an answer is clean, so every
# read of it goes through errs(). Counting None as zero is right; crashing on it is not.
def errs(d):
    return d['validation_errors'] or []

print("Does the no-retrieval arm reproduce the frozen baseline?")
print("(if not, something other than retrieval changed and this A/B is confounded)\n")
confounded = False
for s in ab['none']:
    same_diff = ab['none'][s]['differential'] == base[s]['differential']
    same_esc  = ab['none'][s]['escalation']   == base[s]['escalation']
    confounded |= not (same_diff and same_esc)
    print(f"  {s:<14} differential={'MATCH' if same_diff else 'DIFFERS'}   "
          f"escalation={'MATCH' if same_esc else 'DIFFERS'}")
print("\n  ==> " + ("CONFOUNDED -- investigate before reading anything below"
                     if confounded else "clean: differences below are attributable to retrieval"))

print("\n\nEscalation must be identical across arms -- it is computed before the model runs.")
for s in ab['none']:
    a, b = ab['none'][s]['escalation'], ab['tfidf'][s]['escalation']
    print(f"  {s:<14} {'OK' if a == b else 'VIOLATED -- investigate'}")

print(f"\n\n{'scenario':<16}{'withheld':<14}{'errors':<12}{'warnings':<14}grounded")
print('-' * 72)
for s in ab['none']:
    a, b = ab['none'][s], ab['tfidf'][s]
    wa, wb = a.get('differential_withheld', False), b.get('differential_withheld', False)
    print(f"  {s:<14}{str(wa)[0]} -> {str(wb)[0]:<9}"
          f"{len(errs(a))} -> {len(errs(b)):<8}"
          f"{len(a.get('warnings') or [])} -> {len(b.get('warnings') or []):<10}"
          f"{b['_retrieval']['grounded']}")

print("\n\nErrors that retrieval REMOVED:")
none_removed = True
for s in ab['none']:
    for e in errs(ab['none'][s]):
        if e not in errs(ab['tfidf'][s]):
            print(f"  [{s}] {e}"); none_removed = False
if none_removed:
    print("  (none)")

print("\nErrors retrieval INTRODUCED:")
none_added = True
for s in ab['none']:
    for e in errs(ab['tfidf'][s]):
        if e not in errs(ab['none'][s]):
            print(f"  [{s}] {e}"); none_added = False
if none_added:
    print("  (none)")

## 10. Reproducibility

Temperature 0, fixed seed, KV cache reset per call. If two identical calls diverge, every
difference measured above is noise.

In [ ]:
st = build('missing')
hits = retriever.for_state(st)
a = reason(st, llm_fn=backend, retrieved=hits, max_revisions=1)
b = reason(st, llm_fn=backend, retrieved=hits, max_revisions=1)
print('identical across two calls:', a['differential'] == b['differential'])

## 11. Download

In [ ]:
from google.colab import files
files.download('/content/ab_retrieval.json')